# 02 Split and features

We split students into train / validation / test (70 / 15 / 15), so the test set is made of new students. Then we build the features from earlier answers only.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import load_data, split_students

df = split_students(load_data())
df.shape

(1382727, 20)

## 1. Check the split

In [2]:
print(df.groupby("split")["UserId"].nunique())
print(df["split"].value_counts(normalize=True).round(3))
print("Students in more than one split:", (df.groupby("UserId")["split"].nunique() > 1).sum())
print(df.groupby("split")[["IsCorrect", "PremiumPupil"]].mean().round(3))

split
test      739
train    3442
val       737
Name: UserId, dtype: int64
split
train    0.707
test     0.148
val      0.145
Name: proportion, dtype: float64
Students in more than one split: 0
       IsCorrect  PremiumPupil
split                         
test       0.532         0.303
train      0.536         0.280
val        0.547         0.249


**Takeaway:** 3,442 / 737 / 739 students (70.7% / 14.5% / 14.8% of answers). No student appears in more than one split, so the test set is made of new students. The correct rate is similar in all splits (53.2–54.7%). The free-school-meal share varies a little (24.9–30.3% of known students) by chance, which we note as a limitation for the group comparison.

## 2. Features

Every feature uses only **answers from earlier minutes** by the same student, or statistics from **training students only**.

| Feature | Meaning |
|---|---|
| `n_prev`, `acc_prev` | answers so far and smoothed accuracy so far, over all questions |
| `n_prev_L0`, `acc_prev_L0` | the same, on this question's level-0 topic (Maths). Identical to `acc_prev`, kept only for the level comparison |
| `n_prev_L1`, `acc_prev_L1` | the same, on its level-1 topic (3 topics) |
| `n_prev_L2`, `acc_prev_L2` | the same, on its level-2 skill (25 topics) |
| `n_prev_L3`, `acc_prev_L3` | the same, on its level-3 skill (57 topics) |
| `q_correct_rate` | share of training students who got this question right (leave-one-out for training rows) |

Accuracy is smoothed as (correct + 1) / (answers + 2), so a student with no history starts at 0.5.

In [3]:
from src.features import build_features

df = build_features(df)
df.shape

(1382727, 36)

### Check by hand: the first answers of one student
Answers in the same minute should get the same history, and it should only grow between minutes.

In [4]:
first = df[df["UserId"] == df["UserId"].iloc[0]]
first[["DateAnswered", "QuestionId", "IsCorrect", "n_prev", "correct_prev", "acc_prev"]].head(10)

,DateAnswered,QuestionId,IsCorrect,n_prev,correct_prev,acc_prev
0,2019-10-11 13:15:00,182,0,0,0,0.500000
1,2019-10-11 13:17:00,202,0,1,0,0.333333
2,2019-10-11 13:17:00,420,1,1,0,0.333333
3,2019-10-11 13:18:00,403,1,3,1,0.400000
4,2019-10-11 13:19:00,691,1,4,2,0.500000
5,2019-10-11 13:20:00,261,0,5,3,0.571429
6,2019-10-11 13:20:00,503,0,5,3,0.571429
7,2019-10-11 13:21:00,289,0,7,3,0.444444
8,2019-10-11 13:21:00,566,1,7,3,0.444444
9,2019-10-11 13:22:00,241,1,9,4,0.454545


**Takeaway:** Answers in the same minute share the same history, and the counts only grow from one minute to the next. The earlier-minute rule works.

### Summary of each feature

In [5]:
feature_cols = ["n_prev", "acc_prev", "n_prev_L1", "acc_prev_L1",
                "n_prev_L2", "acc_prev_L2", "n_prev_L3", "acc_prev_L3", "q_correct_rate"]
print("Missing values:", df[feature_cols].isna().sum().sum())
df[feature_cols].describe().round(3)

Missing values: 0


,n_prev,acc_prev,n_prev_L1,acc_prev_L1,n_prev_L2,acc_prev_L2,n_prev_L3,acc_prev_L3,q_correct_rate
count,1382727.000,1382727.000,1382727.000,1382727.000,1382727.000,1382727.000,1382727.000,1382727.000,1382727.000
mean,198.448,0.571,79.862,0.561,24.231,0.559,9.314,0.548,0.536
std,156.837,0.192,67.405,0.201,24.730,0.200,10.505,0.193,0.152
min,0.000,0.053,0.000,0.050,0.000,0.034,0.000,0.038,0.156
25%,69.000,0.419,26.000,0.403,5.000,0.413,2.000,0.425,0.421
50%,160.000,0.561,62.000,0.551,15.000,0.533,6.000,0.500,0.534
75%,295.000,0.707,118.000,0.704,37.000,0.706,13.000,0.667,0.651
max,824.000,0.999,413.000,0.997,115.000,0.991,58.000,0.983,0.932


**Takeaway:** No feature has missing values. The median answer has 160 earlier answers overall, but only 62 on its level-1 topic, 15 on its level-2 skill and 6 on its level-3 skill. The more specific the level, the shorter the history.

### Level 0 is the same as overall history
Every question's level-0 topic is Maths, so level-0 history should equal overall history.

In [6]:
print("acc_prev_L0 identical to acc_prev:", (df["acc_prev_L0"] == df["acc_prev"]).all())
print("n_prev_L0 identical to n_prev:", (df["n_prev_L0"] == df["n_prev"]).all())

acc_prev_L0 identical to acc_prev: True
n_prev_L0 identical to n_prev: True


**Takeaway:** Both are identical, so in the models we use `acc_prev` and `n_prev` and never their level-0 copies. Two identical columns would make logistic regression's weights meaningless.

### Leakage check: does each feature behave the same on train and validation?
If a feature leaked the answer, it would look much better on train than on validation.

In [7]:
from sklearn.metrics import roc_auc_score

features = ["acc_prev_L0", "acc_prev_L1", "acc_prev_L2", "acc_prev_L3", "q_correct_rate"]

rows = []
for col in features:
    row = {"feature": col}
    for s in ["train", "val"]:
        part = df[df["split"] == s]
        row[s] = round(roc_auc_score(part["IsCorrect"], part[col]), 4)
    rows.append(row)
pd.DataFrame(rows)

,feature,train,val
0,acc_prev_L0,0.7006,0.7011
1,acc_prev_L1,0.6991,0.7001
2,acc_prev_L2,0.6829,0.6833
3,acc_prev_L3,0.6647,0.6649
4,q_correct_rate,0.6733,0.6754


**Takeaway:** Train and validation AUC match for every feature (differences ≤ 0.002), so there is no sign of leakage.

On its own, each feature's AUC drops as the skill level gets more specific: L0 (all questions) 0.701, L1 0.700, L2 0.683, L3 0.665. Broader levels have more history, and that matters more than precision for a single feature.

Level-0 history is identical to overall history, since every question's level-0 topic is Maths, so we use only one of them.

Question correct rate alone reaches 0.675, which is the baseline our models must beat.
Whether skill histories add information *on top of* overall accuracy is tested by forward selection in the models notebook.